# Multi-Parameter Analysis

Read multiple parameters at different rates, align them into a single DataFrame, compute statistics, and visualise traces.

**Prerequisites:** Complete [01-getting-started.ipynb](01-getting-started.ipynb) first.

In [ ]:
import sys
sys.path.insert(0, '.')
from sqlrace_helpers import (
    init_sqlrace, load_session, list_parameters,
    extract_parameters, extract_to_timetable, plot_parameters
)
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
SESSION_GUID = "<REPLACE WITH YOUR SESSION GUID>"

sm = init_sqlrace()
client_session, session = load_session(sm, SESSION_GUID)

## Select parameters

Pick parameters to analyse. Here we take up to the first 4.

In [ ]:
all_params = list_parameters(session)
selected = all_params[:4]
print(f"Analysing: {selected}")

## Extract and align

Parameters may be sampled at different rates. `extract_parameters` performs an outer join on timestamps, so faster channels will have NaN in slower channels' rows.

In [ ]:
df = extract_to_timetable(session, selected)
print(f"Shape: {df.shape}")
display(df.head(10))

## Descriptive statistics

Quick summary of each parameter.

In [ ]:
stats = df.describe().T
display(stats)

## Correlation

Check how parameters relate to each other. We forward-fill to handle multi-rate alignment.

In [ ]:
corr = df.ffill().corr()
display(corr.style.background_gradient(cmap="coolwarm", vmin=-1, vmax=1))

## Plot traces

Stacked subplots for visual comparison.

In [ ]:
fig = plot_parameters(df, title=f"Session: {session.Identifier}")
plt.show()

## Scatter plot: parameter vs parameter

If you have two correlated parameters, a scatter plot can reveal their relationship.

In [ ]:
if len(selected) >= 2:
    df_filled = df.ffill().dropna()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df_filled[selected[0]], df_filled[selected[1]],
               alpha=0.3, s=2)
    ax.set_xlabel(selected[0])
    ax.set_ylabel(selected[1])
    ax.set_title(f"{selected[0]} vs {selected[1]}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
client_session.Dispose()
print("Session closed.")